In [1]:
import numpy as np
import torch
from torch import nn
from transformers import AutoModel

class ModelForSourceCodeEmbedding(nn.Module):
    def __init__(self, model_name, normalize=True):
        super(ModelForSourceCodeEmbedding, self).__init__()
        self.model = AutoModel.from_pretrained(model_name, device_map="cuda:0", trust_remote_code=True)
        self.normalize = normalize

    def forward(self, **kwargs):
        model_output = self.model(**kwargs)

        embeddings = model_output.last_hidden_state[:, 0]
        if self.normalize:
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings

    def __getattr__(self, name: str):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.model, name)

In [2]:
import re

def remove_comments(string):
    pattern = r"(\".*?\"|\'.*?\')|(/\*.*?\*/|//[^\r\n]*$)"
    regex = re.compile(pattern, re.MULTILINE|re.DOTALL)
    def _replacer(match):
        if match.group(2) is not None:
            return ""
        else:
            return match.group(1)
    return regex.sub(_replacer, string)


original_code = """
import java.util.Scanner;
public class T7 {
	public static void main(String[] args) {
		Scanner input = new Scanner(System.in);
		System.out.print("Enter a 4 by 4 matrix row by row: ");
		double[][] m = new double[4][4];
		for (int i = 0; i < 4; i++)
			for (int j = 0; j < 4; j++)
				m[i][j] = input.nextDouble();
		System.out.print("Sum of the elements in the major diagonal is " + sumMajorDiagonal(m));
	}
	public static double sumMajorDiagonal(double[][] m) {
		double sum = 0;
		for (int i = 0; i < m.length; i++)
			sum += m[i][i];
		return sum;
	}
}
"""

secondary_code = """
import java.util.Scanner;

public class DiagonalSumCalculator {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        
        // Запрос на ввод данных пользователем
        System.out.println("Введите 4x4 матрицу, вводя элементы построчно: ");
        
        double[][] matrix = new double[4][4];
        
        // Заполнение матрицы с клавиатуры
        for (int row = 0; row < 4; row++) {
            for (int col = 0; col < 4; col++) {
                matrix[row][col] = scanner.nextDouble();
            }
        }
        
        // Вывод результата вычисления суммы главной диагонали
        System.out.println("Сумма элементов на главной диагонали: " + calculateDiagonalSum(matrix));
    }

    public static double calculateDiagonalSum(double[][] array) {
        double diagonalSum = 0;
        
        // Итерация по главной диагонали матрицы
        for (int index = 0; index < array.length; index++) {
            diagonalSum += array[index][index];
        }
        
        return diagonalSum;
    }
}
"""

original_code = remove_comments(original_code)
secondary_code = remove_comments(secondary_code)

In [3]:
from transformers import AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model


model_name = "Salesforce/SFR-Embedding-Code-400M_R"
model = ModelForSourceCodeEmbedding(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    target_modules=["qkv_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

11.8
True
trainable params: 786,432 || all params: 434,925,568 || trainable%: 0.1808199052579038


In [4]:
model.load_state_dict(torch.load(f"models_sfr_embedding_original/12_emb_model.pth", map_location='cpu'))
model.eval()

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): ModelForSourceCodeEmbedding(
      (model): NewModel(
        (embeddings): NewEmbeddings(
          (word_embeddings): Embedding(30528, 1024, padding_idx=0)
          (rotary_emb): NTKScalingRotaryEmbedding()
          (token_type_embeddings): Embedding(2, 1024)
          (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): NewEncoder(
          (layer): ModuleList(
            (0-23): 24 x NewLayer(
              (attention): NewSdpaAttention(
                (qkv_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=3072, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=8, bias=False)
                  )
    

In [5]:
def get_cosing(q1_embs, q2_embs):
    return np.sum(q1_embs * q2_embs)

def threshold(x):
    return 1 if x > 0.5 else 0

In [6]:
import psycopg2


connection = psycopg2.connect(host='localhost', user='postgres', password='12345', dbname='postgres')
cursor = connection.cursor()

In [ ]:
import gradio as gr
import ast


def check_plagiarism(user_code):
    user_code = remove_comments(user_code)
    query_embedding = tokenizer(user_code, return_tensors="pt", max_length=512, truncation=True)
    
    query_embedding = {k: v.to(device) for k, v in query_embedding.items()}
    
    query_embedding = model(**query_embedding)
    query_embedding = query_embedding.cpu().detach().numpy()[0]

    cursor.execute("SELECT * FROM unique_fragments ORDER BY fragment_embedding <=> (%s::vector) LIMIT 1;", (query_embedding.tolist(),))
    result = cursor.fetchone()
    
    if result is not None:
        fragment = result[1]
        fragment_embedding_str = result[2]
        fragment_embedding_list = ast.literal_eval(fragment_embedding_str)
        fragment_embedding = np.array(fragment_embedding_list, dtype=np.float32)
        
        cosine_sim = get_cosing(query_embedding, fragment_embedding)
        prediction = threshold(cosine_sim)
        plagiarism_percent = cosine_sim * 100
    else:
        prediction = 0
        plagiarism_percent = 0
        
    if prediction == 1:
        return f"YOU CODE IS PLAGIARISED\nPLAGIARISM PERCENT: {plagiarism_percent}\nORIGINAL CODE:\n\n{fragment}"
    else:
        cursor.execute("insert into unique_fragments (fragment, fragment_embedding) values (%s, %s)", (user_code, query_embedding.tolist(),))
        connection.commit()
        return "YOUR CODE IS ORIGINAL."


interface = gr.Interface(
    fn=check_plagiarism,
    inputs=gr.Textbox(label="Enter Your Code Here", placeholder="Paste your code here", lines=10),
    outputs=gr.Textbox(label="Result", placeholder="Plagiarism status will appear here", lines=10),
    title="Code Plagiarism Detection",
    description="Check if your code is plagiarized by comparing it with known plagiarized examples.",
)

interface.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
